<a href="https://colab.research.google.com/github/tuckerlucy1/HLS-Data-Resources/blob/main/Qiskit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import json

drive.mount('/content/drive')
# Create a folder for your quantum data
!mkdir -p "/content/drive/My Drive/Quantum_Experiments"


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from google.colab import userdata # For secure token handling

# Modified imports for Estimator V1, removing EstimatorV2 and Session
from qiskit.circuit.library import IQP, XGate
from qiskit.quantum_info import random_hermitian, SparsePauliOp, Statevector
from qiskit_ibm_runtime import QiskitRuntimeService, Estimator # Using Estimator V1
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler.passes.scheduling import ALAPScheduleAnalysis, PadDynamicalDecoupling

# 1. Setup Service (Uses Colab Secrets)
token = userdata.get('Qiskit') # Changed from 'IBM_TOKEN' to 'Qiskit'
service = QiskitRuntimeService(channel="ibm_quantum_platform", token=token) # Corrected channel name
backend = service.least_busy(operational=True, simulator=False)
print(f"Using Backend: {backend.name}")

# 2. Hardware Analysis: Find the cleanest 1D string of qubits
target = backend.target
gate_name = 'cx' if 'cx' in target.operation_names else 'ecr'
gate_errors = []

# Corrected way to access gate properties to avoid TypeError
if gate_name in target.operations:
    operation = target.operations[gate_name]
    if operation.properties: # Check if properties exist for this operation
        for q_tuple, props in operation.properties.items():
            error = props.error
            gate_errors.append((q_tuple, error))
    else:
        print(f"Warning: No properties found for gate '{gate_name}'.")
else:
    print(f"Warning: Gate '{gate_name}' not found in target operations.")


# Sort to find a chain of low-error qubits (simplified logic)
if gate_errors:
    gate_errors.sort(key=lambda x: x[1])
    golden_chain = list(gate_errors[0][0]) # Starting with the best pair
else:
    print("Could not determine golden chain due to missing gate error data.")
    golden_chain = [] # Initialize as empty to prevent further errors

# 3. Scaling Loop Configuration
qubit_range = range(3, 7)
results = {"qubits": [], "hw_evs": [], "theo_evs": [], "stds": []}

# Options for Estimator V1, passed as a dictionary
# Note: Estimator V1 options are structured differently than EstimatorOptions for V2
runtime_options = {
    "resilience_level": 2, # Readout + Bias mitigation
    "shots": 8192
}

# Instantiate Estimator V1 directly with service and options
estimator = Estimator(service=service, options=runtime_options)

# 4. Run the Experiment
# No 'with Session(...)' block for Estimator V1 when used this way
for n in qubit_range:
    print(f"Running n={n}...")

    # A. Create IQP and Theory
    mat = np.real(random_hermitian(n, seed=42))
    qc = IQP(mat)
    obs = SparsePauliOp("Z" * n)
    theo_ev = Statevector.from_instruction(qc).expectation_value(obs).real

    # B. Hardware-Aware Pass Manager
    pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
    durations = backend.target.durations()

    # ADD DYNAMICAL DECOUPLING (The Shield)
    pm.add_pass(ALAPScheduleAnalysis(durations))
    pm.add_pass(PadDynamicalDecoupling(durations, [XGate(), XGate()]))

    isa_qc = pm.run(qc)
    isa_obs = obs.apply_layout(isa_qc.layout)

    # C. Execute with Estimator V1
    # Estimator V1 run method call
    job = estimator.run(isa_qc, observables=isa_obs)
    result = job.result() # Get the PrimitiveResult object
    pub_res = result.pub_results[0] # Access results for the first publication

    results["qubits"].append(n)
    results["hw_evs"].append(pub_res.data.evs)
    results["theo_evs"].append(theo_ev)
    results["stds"].append(pub_res.data.stds)

# --- STEP 3: PLOTTING ---
plt.errorbar(results["qubits"], results["hw_evs"], yerr=results["stds"], fmt='-o', label='Hardware')
plt.plot(results["qubits"], results["theo_evs"], 'r--', label='Ideal')
plt.title(f"IQP Scaling on {backend.name}")
plt.xlabel("Qubits")
plt.ylabel("Expectation Value")
plt.legend()
plt.show()

### Refined Plotting of Experimental Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create the plot
plt.figure(figsize=(10, 6)) # Adjust figure size for better visualization
plt.errorbar(results["qubits"], results["hw_evs"], yerr=results["stds"], fmt='-o', capsize=5, label='Hardware Expectation Value')
plt.plot(results["qubits"], results["theo_evs"], 'r--', label='Ideal Expectation Value')

# Add more descriptive labels and title
plt.title(f"IQP Scaling Experiment on {backend.name}\nHardware vs. Ideal Expectation Value", fontsize=14)
plt.xlabel("Number of Qubits (n)", fontsize=12)
plt.ylabel("Expectation Value <Z^n>", fontsize=12)
plt.xticks(results["qubits"]) # Ensure x-axis ticks correspond to qubit numbers
plt.grid(True, linestyle='--', alpha=0.7) # Add a grid for readability
plt.legend(fontsize=10)
plt.tight_layout() # Adjust layout to prevent labels from overlapping

# Save the plot to Google Drive
plot_file_path = f"/content/drive/My Drive/Quantum_Experiments/IQP_Scaling_Plot_{backend.name}.png"
plt.savefig(plot_file_path)
print(f"✅ Plot saved to your Drive at: {plot_file_path}")

plt.show()

In [ ]:
from google.colab import userdata

token_qiskit = userdata.get('Qiskit')

if token_qiskit:
    print(f"Secret 'Qiskit' retrieved successfully: {token_qiskit}")
else:
    print("Secret 'Qiskit' not found or empty in secrets.")

In [ ]:
from google.colab import userdata

token_qiskit = userdata.get('Qiskit')

if token_qiskit:
    print(f"Secret 'Qiskit' retrieved successfully: {token_qiskit}")
else:
    print("Secret 'Qiskit' not found or empty in secrets.")

In [ ]:
from google.colab import userdata
token = userdata.get('IBM_TOKEN')
if token:
    print("IBM_TOKEN successfully retrieved from secrets!")
else:
    print("IBM_TOKEN not found or empty in secrets.")

In [ ]:
file_path = f"/content/drive/My Drive/Quantum_Experiments/IQP_Scaling_{backend.name}.json"

with open(file_path, 'w') as f:
    # We convert numpy arrays to lists so JSON can handle them
    json_ready_results = {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in results.items()}
    json.dump(json_ready_results, f)

print(f"✅ Data safely tucked away in your Drive at: {file_path}")


# New Section

# Task
First, correct the `TypeError` in `cell_id: OxF4VQLaUJZx` by modifying the `Estimator` instantiation to `estimator = Estimator(backend=backend, options=runtime_options)`, aligning it with Qiskit Runtime V2 API. Second, enhance the qubit selection logic in the same cell: replace the failing `target.operations` approach with one that uses `backend.properties().gates` to accurately collect two-qubit gate (CX/ECR) error rates. Identify a set of highly performing qubits based on these errors, populating the `golden_chain` variable to address the previous warning and lay the groundwork for refined qubit selection, then execute the modified cell.

## Refine Qubit Selection

### Subtask:
Correct the `TypeError` in the `Estimator` instantiation and enhance the qubit selection logic by using `backend.properties().gates` to collect two-qubit gate error rates, then identify a highly performing qubit chain.


**Reasoning**:
The subtask requires correcting a TypeError in the `Estimator` instantiation and refining the qubit selection logic. I will provide a `code_block` that modifies the existing cell to address these issues, as per the instructions.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from google.colab import userdata # For secure token handling

# Modified imports for Estimator V1, removing EstimatorV2 and Session
from qiskit.circuit.library import IQP, XGate
from qiskit.quantum_info import random_hermitian, SparsePauliOp, Statevector
from qiskit_ibm_runtime import QiskitRuntimeService, Estimator # Using Estimator V1
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler.passes.scheduling import ALAPScheduleAnalysis, PadDynamicalDecoupling

# 1. Setup Service (Uses Colab Secrets)
token = userdata.get('Qiskit') # Changed from 'IBM_TOKEN' to 'Qiskit'
service = QiskitRuntimeService(channel="ibm_quantum_platform", token=token) # Corrected channel name
backend = service.least_busy(operational=True, simulator=False)
print(f"Using Backend: {backend.name}")

# 2. Hardware Analysis: Find the cleanest 1D string of qubits
target = backend.target
gate_errors = []

# --- MODIFIED QUBIT SELECTION LOGIC ---
props = backend.properties()

# Determine the primary two-qubit gate
if props and props.gates:
    if 'cx' in props.gates:
        gate_name = 'cx'
    elif 'ecr' in props.gates:
        gate_name = 'ecr'
    else:
        gate_name = None
        print("Warning: Neither 'cx' nor 'ecr' found in backend properties gates.")
else:
    gate_name = None
    print("Warning: Backend properties or gates not available.")


if gate_name:
    # Iterate through the gate properties for the identified gate name
    for q_tuple, gate_prop in props.gates[gate_name].items():
        if 'error' in gate_prop:
            error = gate_prop['error']
            gate_errors.append((q_tuple, error))

    # Sort to find a chain of low-error qubits
    if gate_errors:
        gate_errors.sort(key=lambda x: x[1])
        golden_chain = list(gate_errors[0][0]) # Starting with the best pair
        print(f"Identified best {gate_name} gate: {gate_errors[0][0]} with error: {gate_errors[0][1]:.4f}")
        print(f"Golden chain (best initial pair): {golden_chain}")
    else:
        print(f"Could not determine golden chain due to missing {gate_name} gate error data.")
        golden_chain = [] # Initialize as empty to prevent further errors
else:
    print("Could not determine golden chain as no suitable two-qubit gate was found.")
    golden_chain = []
# --- END MODIFIED QUBIT SELECTION LOGIC ---

# 3. Scaling Loop Configuration
qubit_range = range(3, 7)
results = {"qubits": [], "hw_evs": [], "theo_evs": [], "stds": []}

# Options for Estimator V1, passed as a dictionary
# Note: Estimator V1 options are structured differently than EstimatorOptions for V2
runtime_options = {
    "resilience_level": 2, # Readout + Bias mitigation
    "shots": 8192
}

# Instantiate Estimator V1 directly with backend and options
estimator = Estimator(backend=backend, options=runtime_options)

# 4. Run the Experiment
# No 'with Session(...)' block for Estimator V1 when used this way
for n in qubit_range:
    print(f"Running n={n}...")

    # A. Create IQP and Theory
    mat = np.real(random_hermitian(n, seed=42))
    qc = IQP(mat)
    obs = SparsePauliOp("Z" * n)
    theo_ev = Statevector.from_instruction(qc).expectation_value(obs).real

    # B. Hardware-Aware Pass Manager
    pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
    durations = backend.target.durations()

    # ADD DYNAMICAL DECOUPLING (The Shield)
    pm.add_pass(ALAPScheduleAnalysis(durations))
    pm.add_pass(PadDynamicalDecoupling(durations, [XGate(), XGate()]))

    isa_qc = pm.run(qc)
    isa_obs = obs.apply_layout(isa_qc.layout)

    # C. Execute with Estimator V1
    # Estimator V1 run method call
    job = estimator.run(isa_qc, observables=isa_obs)
    result = job.result() # Get the PrimitiveResult object
    pub_res = result.pub_results[0] # Access results for the first publication

    results["qubits"].append(n)
    results["hw_evs"].append(pub_res.data.evs)
    results["theo_evs"].append(theo_ev)
    results["stds"].append(pub_res.data.stds)

# --- STEP 3: PLOTTING ---
plt.errorbar(results["qubits"], results["hw_evs"], yerr=results["stds"], fmt='-o', label='Hardware')
plt.plot(results["qubits"], results["theo_evs"], 'r--', label='Ideal')
plt.title(f"IQP Scaling on {backend.name}")
plt.xlabel("Qubits")
plt.ylabel("Expectation Value")
plt.legend()
plt.show()

**Reasoning**:
The previous attempt to instantiate `Estimator` resulted in a `TypeError` because `EstimatorV2` does not accept a `backend` argument in its constructor. Additionally, the qubit selection logic failed to correctly parse backend properties. This revised code will instantiate `Estimator` with the `service` object and a proper `EstimatorOptions` object, aligning with Qiskit Runtime V2 API. It also updates the qubit selection logic to correctly iterate through `backend.properties().gates` to identify the lowest-error two-qubit gate and populate the `golden_chain` variable.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from google.colab import userdata # For secure token handling

# Modified imports for Estimator V2
from qiskit.circuit.library import IQP, XGate
from qiskit.quantum_info import random_hermitian, SparsePauliOp, Statevector
from qiskit_ibm_runtime import QiskitRuntimeService, Estimator, EstimatorOptions # Using Estimator and EstimatorOptions for V2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler.passes.scheduling import ALAPScheduleAnalysis, PadDynamicalDecoupling

# 1. Setup Service (Uses Colab Secrets)
token = userdata.get('Qiskit')
service = QiskitRuntimeService(channel="ibm_quantum_platform", token=token)
backend = service.least_busy(operational=True, simulator=False)
print(f"Using Backend: {backend.name}")

# 2. Hardware Analysis: Find the cleanest 1D string of qubits
props = backend.properties() # Get BackendProperties object
gate_errors = []
golden_chain = [] # Initialize golden_chain

if props:
    # Determine the primary two-qubit gate available
    # Check if 'cx' or 'ecr' exist in props.gates (which is a dict of lists of Gate objects)
    gate_name = None
    if 'cx' in props.gates:
        gate_name = 'cx'
    elif 'ecr' in props.gates:
        gate_name = 'ecr'

    if gate_name:
        # props.gates[gate_name] is a list of Gate objects
        for gate_obj in props.gates[gate_name]:
            qubits = tuple(gate_obj.qubits)
            error = None
            for param in gate_obj.parameters:
                if param.name == 'error':
                    error = param.value
                    break
            if error is not None:
                gate_errors.append((qubits, error))

        # Sort to find a chain of low-error qubits
        if gate_errors:
            gate_errors.sort(key=lambda x: x[1])
            golden_chain = list(gate_errors[0][0]) # Starting with the best pair
            print(f"Identified best {gate_name} gate: {gate_errors[0][0]} with error: {gate_errors[0][1]:.4f}")
            print(f"Golden chain (best initial pair): {golden_chain}")
        else:
            print(f"Could not determine golden chain due to missing {gate_name} gate error data.")
    else:
        print("Warning: Neither 'cx' nor 'ecr' found in backend properties gates.")
else:
    print("Warning: Backend properties not available.")

# 3. Scaling Loop Configuration
qubit_range = range(3, 7)
results = {"qubits": [], "hw_evs": [], "theo_evs": [], "stds": []}

# Options for Estimator V2, passed as an EstimatorOptions object
runtime_options_dict = {
    "resilience_level": 2, # Readout + Bias mitigation
    "shots": 8192
}
estimator_options = EstimatorOptions(**runtime_options_dict)

# Instantiate Estimator V2 with service and options
estimator = Estimator(service=service, options=estimator_options)

# 4. Run the Experiment
for n in qubit_range:
    print(f"Running n={n}...")

    # A. Create IQP and Theory
    mat = np.real(random_hermitian(n, seed=42))
    qc = IQP(mat)
    obs = SparsePauliOp("Z" * n)
    theo_ev = Statevector.from_instruction(qc).expectation_value(obs).real

    # B. Hardware-Aware Pass Manager
    pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
    durations = backend.target.durations()

    # ADD DYNAMICAL DECOUPLING (The Shield)
    pm.add_pass(ALAPScheduleAnalysis(durations))
    pm.add_pass(PadDynamicalDecoupling(durations, [XGate(), XGate()]))

    isa_qc = pm.run(qc)
    # Apply layout to observables if transpiler changed it
    if isa_qc.layout:
        isa_obs = obs.apply_layout(isa_qc.layout)
    else:
        isa_obs = obs # Use original if no layout was applied

    # C. Execute with Estimator V2
    # Estimator V2 run method call. Pass backend_name explicitly to ensure it runs on the selected backend.
    job = estimator.run(isa_qc, observables=isa_obs, backend_name=backend.name)
    result = job.result() # Get the PrimitiveResult object (list of results for V2)
    pub_res = result[0] # Access results for the first publication in V2 results

    results["qubits"].append(n)
    results["hw_evs"].append(pub_res.data.evs[0]) # V2 result data is typically a numpy array of results
    results["theo_evs"].append(theo_ev)
    results["stds"].append(pub_res.data.stds[0]) # V2 result data is typically a numpy array of stds

# --- STEP 3: PLOTTING ---
plt.errorbar(results["qubits"], results["hw_evs"], yerr=results["stds"], fmt='-o', label='Hardware')
plt.plot(results["qubits"], results["theo_evs"], 'r--', label='Ideal')
plt.title(f"IQP Scaling on {backend.name}")
plt.xlabel("Qubits")
plt.ylabel("Expectation Value")
plt.legend()
plt.show()

**Reasoning**:
The `ValidationError` arose because `EstimatorOptions` in Qiskit Runtime V2 does not accept `shots` as a direct keyword argument. Instead, `shots` must be set via its `run_options` attribute. Additionally, the qubit selection logic is updated to correctly configure the `EstimatorOptions` for V2 and correctly parse `backend.properties().gates` for two-qubit gate errors.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from google.colab import userdata # For secure token handling

# Modified imports for Estimator V2
from qiskit.circuit.library import IQP, XGate
from qiskit.quantum_info import random_hermitian, SparsePauliOp, Statevector
from qiskit_ibm_runtime import QiskitRuntimeService, Estimator, EstimatorOptions # Using Estimator and EstimatorOptions for V2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler.passes.scheduling import ALAPScheduleAnalysis, PadDynamicalDecoupling

# 1. Setup Service (Uses Colab Secrets)
token = userdata.get('Qiskit')
service = QiskitRuntimeService(channel="ibm_quantum_platform", token=token)
backend = service.least_busy(operational=True, simulator=False)
print(f"Using Backend: {backend.name}")

# 2. Hardware Analysis: Find the cleanest 1D string of qubits
props = backend.properties() # Get BackendProperties object
gate_errors = []
golden_chain = [] # Initialize golden_chain

if props:
    # Determine the primary two-qubit gate available
    # Check if 'cx' or 'ecr' exist in props.gates (which is a dict of lists of Gate objects)
    gate_name = None
    if 'cx' in props.gates:
        gate_name = 'cx'
    elif 'ecr' in props.gates:
        gate_name = 'ecr'

    if gate_name:
        # props.gates[gate_name] is a list of Gate objects
        for gate_obj in props.gates[gate_name]:
            qubits = tuple(gate_obj.qubits)
            error = None
            for param in gate_obj.parameters:
                if param.name == 'error':
                    error = param.value
                    break
            if error is not None:
                gate_errors.append((qubits, error))

        # Sort to find a chain of low-error qubits
        if gate_errors:
            gate_errors.sort(key=lambda x: x[1])
            golden_chain = list(gate_errors[0][0]) # Starting with the best pair
            print(f"Identified best {gate_name} gate: {gate_errors[0][0]} with error: {gate_errors[0][1]:.4f}")
            print(f"Golden chain (best initial pair): {golden_chain}")
        else:
            print(f"Could not determine golden chain due to missing {gate_name} gate error data.")
    else:
        print("Warning: Neither 'cx' nor 'ecr' found in backend properties gates.")
else:
    print("Warning: Backend properties not available.")

# 3. Scaling Loop Configuration
qubit_range = range(3, 7)
results = {"qubits": [], "hw_evs": [], "theo_evs": [], "stds": []}

# Options for Estimator V2, passed as an EstimatorOptions object
estimator_options = EstimatorOptions(
    resilience_level=2 # resilience_level is directly supported
)
estimator_options.run_options.shots = 8192 # shots is set via run_options

# Instantiate Estimator V2 with service and options
estimator = Estimator(service=service, options=estimator_options)

# 4. Run the Experiment
for n in qubit_range:
    print(f"Running n={n}...")

    # A. Create IQP and Theory
    mat = np.real(random_hermitian(n, seed=42))
    qc = IQP(mat)
    obs = SparsePauliOp("Z" * n)
    theo_ev = Statevector.from_instruction(qc).expectation_value(obs).real

    # B. Hardware-Aware Pass Manager
    pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
    durations = backend.target.durations()

    # ADD DYNAMICAL DECOUPLING (The Shield)
    pm.add_pass(ALAPScheduleAnalysis(durations))
    pm.add_pass(PadDynamicalDecoupling(durations, [XGate(), XGate()]))

    isa_qc = pm.run(qc)
    # Apply layout to observables if transpiler changed it
    if isa_qc.layout:
        isa_obs = obs.apply_layout(isa_qc.layout)
    else:
        isa_obs = obs # Use original if no layout was applied

    # C. Execute with Estimator V2
    # Estimator V2 run method call. Pass backend_name explicitly to ensure it runs on the selected backend.
    job = estimator.run(isa_qc, observables=isa_obs, backend_name=backend.name)
    result = job.result() # Get the PrimitiveResult object (list of results for V2)
    pub_res = result[0] # Access results for the first publication in V2 results

    results["qubits"].append(n)
    results["hw_evs"].append(pub_res.data.evs[0]) # V2 result data is typically a numpy array of results
    results["theo_evs"].append(theo_ev)
    results["stds"].append(pub_res.data.stds[0]) # V2 result data is typically a numpy array of stds

# --- STEP 3: PLOTTING ---
plt.errorbar(results["qubits"], results["hw_evs"], yerr=results["stds"], fmt='-o', label='Hardware')
plt.plot(results["qubits"], results["theo_evs"], 'r--', label='Ideal')
plt.title(f"IQP Scaling on {backend.name}")
plt.xlabel("Qubits")
plt.ylabel("Expectation Value")
plt.legend()
plt.show()


**Reasoning**:
The `AttributeError` indicates that `EstimatorOptions` in Qiskit Runtime V2 does not have a `run_options` attribute for setting shots. Instead, `shots` should be passed directly as a keyword argument to the `estimator.run()` method. This modification aligns with the correct usage of the Qiskit Runtime V2 Estimator API.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from google.colab import userdata # For secure token handling

# Modified imports for Estimator V2
from qiskit.circuit.library import IQP, XGate
from qiskit.quantum_info import random_hermitian, SparsePauliOp, Statevector
from qiskit_ibm_runtime import QiskitRuntimeService, Estimator, EstimatorOptions # Using Estimator and EstimatorOptions for V2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler.passes.scheduling import ALAPScheduleAnalysis, PadDynamicalDecoupling

# 1. Setup Service (Uses Colab Secrets)
token = userdata.get('Qiskit')
service = QiskitRuntimeService(channel="ibm_quantum_platform", token=token)
backend = service.least_busy(operational=True, simulator=False)
print(f"Using Backend: {backend.name}")

# 2. Hardware Analysis: Find the cleanest 1D string of qubits
props = backend.properties() # Get BackendProperties object
gate_errors = []
golden_chain = [] # Initialize golden_chain

if props:
    # Determine the primary two-qubit gate available
    # Check if 'cx' or 'ecr' exist in props.gates (which is a dict of lists of Gate objects)
    gate_name = None
    if 'cx' in props.gates:
        gate_name = 'cx'
    elif 'ecr' in props.gates:
        gate_name = 'ecr'

    if gate_name:
        # props.gates[gate_name] is a list of Gate objects
        for gate_obj in props.gates[gate_name]:
            qubits = tuple(gate_obj.qubits)
            error = None
            for param in gate_obj.parameters:
                if param.name == 'error':
                    error = param.value
                    break
            if error is not None:
                gate_errors.append((qubits, error))

        # Sort to find a chain of low-error qubits
        if gate_errors:
            gate_errors.sort(key=lambda x: x[1])
            golden_chain = list(gate_errors[0][0]) # Starting with the best pair
            print(f"Identified best {gate_name} gate: {gate_errors[0][0]} with error: {gate_errors[0][1]:.4f}")
            print(f"Golden chain (best initial pair): {golden_chain}")
        else:
            print(f"Could not determine golden chain due to missing {gate_name} gate error data.")
    else:
        print("Warning: Neither 'cx' nor 'ecr' found in backend properties gates.")
else:
    print("Warning: Backend properties not available.")

# 3. Scaling Loop Configuration
qubit_range = range(3, 7)
results = {"qubits": [], "hw_evs": [], "theo_evs": [], "stds": []}

# Options for Estimator V2, passed as an EstimatorOptions object
estimator_options = EstimatorOptions(
    resilience_level=2 # resilience_level is directly supported
)
# Removed: estimator_options.run_options.shots = 8192 # shots is set via run_options

# Instantiate Estimator V2 with service and options
estimator = Estimator(service=service, options=estimator_options)

# 4. Run the Experiment
for n in qubit_range:
    print(f"Running n={n}...")

    # A. Create IQP and Theory
    mat = np.real(random_hermitian(n, seed=42))
    qc = IQP(mat)
    obs = SparsePauliOp("Z" * n)
    theo_ev = Statevector.from_instruction(qc).expectation_value(obs).real

    # B. Hardware-Aware Pass Manager
    pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
    durations = backend.target.durations()

    # ADD DYNAMICAL DECOUPLING (The Shield)
    pm.add_pass(ALAPScheduleAnalysis(durations))
    pm.add_pass(PadDynamicalDecoupling(durations, [XGate(), XGate()]))

    isa_qc = pm.run(qc)
    # Apply layout to observables if transpiler changed it
    if isa_qc.layout:
        isa_obs = obs.apply_layout(isa_qc.layout)
    else:
        isa_obs = obs # Use original if no layout was applied

    # C. Execute with Estimator V2
    # Estimator V2 run method call. Pass backend_name explicitly to ensure it runs on the selected backend.
    # Shots is now passed directly to the run method.
    job = estimator.run(isa_qc, observables=isa_obs, backend_name=backend.name, shots=8192)
    result = job.result() # Get the PrimitiveResult object (list of results for V2)
    pub_res = result[0] # Access results for the first publication in V2 results

    results["qubits"].append(n)
    results["hw_evs"].append(pub_res.data.evs[0]) # V2 result data is typically a numpy array of results
    results["theo_evs"].append(theo_ev)
    results["stds"].append(pub_res.data.stds[0]) # V2 result data is typically a numpy array of stds

# --- STEP 3: PLOTTING ---
plt.errorbar(results["qubits"], results["hw_evs"], yerr=results["stds"], fmt='-o', label='Hardware')
plt.plot(results["qubits"], results["theo_evs"], 'r--', label='Ideal')
plt.title(f"IQP Scaling on {backend.name}")
plt.xlabel("Qubits")
plt.ylabel("Expectation Value")
plt.legend()
plt.show()


**Reasoning**:
The `TypeError` in `EstimatorV2.__init__()` indicates that neither `service` nor `backend` should be passed to its constructor. Instead, `EstimatorV2` should be instantiated with `options` only. Additionally, the qubit selection logic for `backend.properties().gates` is corrected to iterate over a list of `Gate` objects, rather than a dictionary, to correctly identify two-qubit gate errors and populate the `golden_chain`.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from google.colab import userdata # For secure token handling

# Modified imports for Estimator V2
from qiskit.circuit.library import IQP, XGate
from qiskit.quantum_info import random_hermitian, SparsePauliOp, Statevector
from qiskit_ibm_runtime import QiskitRuntimeService, Estimator, EstimatorOptions # Using Estimator and EstimatorOptions for V2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler.passes.scheduling import ALAPScheduleAnalysis, PadDynamicalDecoupling

# 1. Setup Service (Uses Colab Secrets)
token = userdata.get('Qiskit')
service = QiskitRuntimeService(channel="ibm_quantum_platform", token=token)
backend = service.least_busy(operational=True, simulator=False)
print(f"Using Backend: {backend.name}")

# 2. Hardware Analysis: Find the cleanest 1D string of qubits
props = backend.properties() # Get BackendProperties object
gate_errors = []
golden_chain = [] # Initialize golden_chain

# --- REVISED QUBIT SELECTION LOGIC ---
if props and props.gates:
    found_two_qubit_gate = False
    for gate_obj in props.gates: # Iterate directly over the list of Gate objects
        if gate_obj.gate in ['cx', 'ecr'] and len(gate_obj.qubits) == 2:
            found_two_qubit_gate = True
            qubits = tuple(gate_obj.qubits)
            error = None
            for param in gate_obj.parameters:
                if param.name == 'error':
                    error = param.value
                    break
            if error is not None:
                gate_errors.append((qubits, error))

    if gate_errors:
        gate_errors.sort(key=lambda x: x[1])
        golden_chain = list(gate_errors[0][0]) # Starting with the best pair
        print(f"Identified best two-qubit gate: {gate_errors[0][0]} with error: {gate_errors[0][1]:.4f}")
        print(f"Golden chain (best initial pair): {golden_chain}")
    else:
        # This case is if two-qubit gates were found but had no error data
        if found_two_qubit_gate:
            print("Could not determine golden chain due to missing error data for two-qubit gates.")
        else:
            print("No suitable two-qubit gates ('cx' or 'ecr') found in backend properties.")
        golden_chain = [] # Ensure golden_chain is empty if not determined
else:
    print("Warning: Backend properties or gates not available.")
# --- END REVISED QUBIT SELECTION LOGIC ---

# 3. Scaling Loop Configuration
qubit_range = range(3, 7)
results = {"qubits": [], "hw_evs": [], "theo_evs": [], "stds": []}

# Options for Estimator V2, passed as an EstimatorOptions object
estimator_options = EstimatorOptions(
    resilience_level=2 # resilience_level is directly supported
)

# Instantiate Estimator V2 with options (removed service=service)
estimator = Estimator(options=estimator_options)

# 4. Run the Experiment
for n in qubit_range:
    print(f"Running n={n}...")

    # A. Create IQP and Theory
    mat = np.real(random_hermitian(n, seed=42))
    qc = IQP(mat)
    obs = SparsePauliOp("Z" * n)
    theo_ev = Statevector.from_instruction(qc).expectation_value(obs).real

    # B. Hardware-Aware Pass Manager
    pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
    durations = backend.target.durations()

    # ADD DYNAMICAL DECOUPLING (The Shield)
    pm.add_pass(ALAPScheduleAnalysis(durations))
    pm.add_pass(PadDynamicalDecoupling(durations, [XGate(), XGate()]))

    isa_qc = pm.run(qc)
    # Apply layout to observables if transpiler changed it
    if isa_qc.layout:
        isa_obs = obs.apply_layout(isa_qc.layout)
    else:
        isa_obs = obs # Use original if no layout was applied

    # C. Execute with Estimator V2
    # Estimator V2 run method call. Pass backend_name explicitly to ensure it runs on the selected backend.
    # Shots is now passed directly to the run method.
    job = estimator.run(isa_qc, observables=isa_obs, backend_name=backend.name, shots=8192)
    result = job.result() # Get the PrimitiveResult object (list of results for V2)
    pub_res = result[0] # Access results for the first publication in V2 results

    results["qubits"].append(n)
    results["hw_evs"].append(pub_res.data.evs[0]) # V2 result data is typically a numpy array of results
    results["theo_evs"].append(theo_ev)
    results["stds"].append(pub_res.data.stds[0]) # V2 result data is typically a numpy array of stds

# --- STEP 3: PLOTTING ---
plt.errorbar(results["qubits"], results["hw_evs"], yerr=results["stds"], fmt='-o', label='Hardware')
plt.plot(results["qubits"], results["theo_evs"], 'r--', label='Ideal')
plt.title(f"IQP Scaling on {backend.name}")
plt.xlabel("Qubits")
plt.ylabel("Expectation Value")
plt.legend()
plt.show()

**Reasoning**:
The `ValueError` indicates that the `Estimator` (V2) constructor requires a `mode` argument (either a `Backend` or `Session` object) when instantiated. I will provide the `backend` object as the `mode` to resolve this error. The qubit selection logic remains robust for handling `backend.properties().gates`.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from google.colab import userdata # For secure token handling

# Modified imports for Estimator V2
from qiskit.circuit.library import IQP, XGate
from qiskit.quantum_info import random_hermitian, SparsePauliOp, Statevector
from qiskit_ibm_runtime import QiskitRuntimeService, Estimator, EstimatorOptions # Using Estimator and EstimatorOptions for V2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler.passes.scheduling import ALAPScheduleAnalysis, PadDynamicalDecoupling

# 1. Setup Service (Uses Colab Secrets)
token = userdata.get('Qiskit')
service = QiskitRuntimeService(channel="ibm_quantum_platform", token=token)
backend = service.least_busy(operational=True, simulator=False)
print(f"Using Backend: {backend.name}")

# 2. Hardware Analysis: Find the cleanest 1D string of qubits
props = backend.properties() # Get BackendProperties object
gate_errors = []
golden_chain = [] # Initialize golden_chain

# --- REVISED QUBIT SELECTION LOGIC ---
if props and props.gates:
    found_two_qubit_gate = False
    for gate_obj in props.gates: # Iterate directly over the list of Gate objects
        if gate_obj.gate in ['cx', 'ecr'] and len(gate_obj.qubits) == 2:
            found_two_qubit_gate = True
            qubits = tuple(gate_obj.qubits)
            error = None
            for param in gate_obj.parameters:
                if param.name == 'error':
                    error = param.value
                    break
            if error is not None:
                gate_errors.append((qubits, error))

    if gate_errors:
        gate_errors.sort(key=lambda x: x[1])
        golden_chain = list(gate_errors[0][0]) # Starting with the best pair
        print(f"Identified best two-qubit gate: {gate_errors[0][0]} with error: {gate_errors[0][1]:.4f}")
        print(f"Golden chain (best initial pair): {golden_chain}")
    else:
        # This case is if two-qubit gates were found but had no error data
        if found_two_qubit_gate:
            print("Could not determine golden chain due to missing error data for two-qubit gates.")
        else:
            print("No suitable two-qubit gates ('cx' or 'ecr') found in backend properties.")
        golden_chain = [] # Ensure golden_chain is empty if not determined
else:
    print("Warning: Backend properties or gates not available.")
# --- END REVISED QUBIT SELECTION LOGIC ---

# 3. Scaling Loop Configuration
qubit_range = range(3, 7)
results = {"qubits": [], "hw_evs": [], "theo_evs": [], "stds": []}

# Options for Estimator V2, passed as an EstimatorOptions object
estimator_options = EstimatorOptions(
    resilience_level=2 # resilience_level is directly supported
)

# Instantiate Estimator V2 with options, providing the backend as the mode
estimator = Estimator(mode=backend, options=estimator_options)

# 4. Run the Experiment
for n in qubit_range:
    print(f"Running n={n}...")

    # A. Create IQP and Theory
    mat = np.real(random_hermitian(n, seed=42))
    qc = IQP(mat)
    obs = SparsePauliOp("Z" * n)
    theo_ev = Statevector.from_instruction(qc).expectation_value(obs).real

    # B. Hardware-Aware Pass Manager
    pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
    durations = backend.target.durations()

    # ADD DYNAMICAL DECOUPLING (The Shield)
    pm.add_pass(ALAPScheduleAnalysis(durations))
    pm.add_pass(PadDynamicalDecoupling(durations, [XGate(), XGate()]))

    isa_qc = pm.run(qc)
    # Apply layout to observables if transpiler changed it
    if isa_qc.layout:
        isa_obs = obs.apply_layout(isa_qc.layout)
    else:
        isa_obs = obs # Use original if no layout was applied

    # C. Execute with Estimator V2
    # Estimator V2 run method call. Pass backend_name explicitly to ensure it runs on the selected backend.
    # Shots is passed directly to the run method.
    job = estimator.run(isa_qc, observables=isa_obs, backend_name=backend.name, shots=8192)
    result = job.result() # Get the PrimitiveResult object (list of results for V2)
    pub_res = result[0] # Access results for the first publication in V2 results

    results["qubits"].append(n)
    results["hw_evs"].append(pub_res.data.evs[0]) # V2 result data is typically a numpy array of results
    results["theo_evs"].append(theo_ev)
    results["stds"].append(pub_res.data.stds[0]) # V2 result data is typically a numpy array of stds

# --- STEP 3: PLOTTING ---
plt.errorbar(results["qubits"], results["hw_evs"], yerr=results["stds"], fmt='-o', label='Hardware')
plt.plot(results["qubits"], results["theo_evs"], 'r--', label='Ideal')
plt.title(f"IQP Scaling on {backend.name}")
plt.xlabel("Qubits")
plt.ylabel("Expectation Value")
plt.legend()
plt.show()